# SQL Inference — Llama-3.2-3B + LoRA (vLLM)
**Model:** `meta-llama/Llama-3.2-3B-Instruct`  
**Adapter:** `glen-louis/llama-3.2-3b-sql-qlora`  

Run on Colab with GPU runtime: `Runtime → Change runtime type → T4 GPU`

## 1. Check GPU

In [ ]:
!nvidia-smi

## 2. Install Dependencies

In [ ]:
!pip install transformers peft accelerate bitsandbytes -q

## 3. Authenticate with HuggingFace

Required to download `meta-llama/Llama-3.2-3B-Instruct` (gated model).

In [ ]:
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

## 4. Load Model + LoRA Adapter

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL = "meta-llama/Llama-3.2-3B-Instruct"
ADAPTER    = "glen-louis/llama-3.2-3b-sql-qlora"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
)

model = PeftModel.from_pretrained(base_model, ADAPTER)
model.eval()

print("Model loaded.")

## 5. Prompt Formatter

Matches the format used during fine-tuning (`data_utils.py`).

In [ ]:
def build_prompt(question: str, schema: str) -> str:
    messages = [
        {
            "role": "system",
            "content": (
                "You are a SQL expert. Given a natural language question and a database schema, "
                "write a SQL query that answers the question. Return only the SQL query with no explanation."
            ),
        },
        {
            "role": "user",
            "content": f"{question}\n\nSchema:\n{schema}",
        },
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

## 6. Run Inference

In [ ]:
def generate_sql(question: str, schema: str, max_new_tokens: int = 256) -> str:
    prompt = build_prompt(question, schema)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    # Decode only the newly generated tokens
    new_tokens = output_ids[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


# --- Example ---
question = "What are the top 5 customers by total order value?"

schema = """
CREATE TABLE customers (id INT, name TEXT, email TEXT);
CREATE TABLE orders (id INT, customer_id INT, amount DECIMAL, order_date DATE);
"""

print(generate_sql(question, schema))

## 7. Batch Inference (optional)

Generate SQL for multiple questions at once — vLLM handles batching efficiently.

In [ ]:
examples = [
    {
        "question": "How many orders were placed in 2023?",
        "schema": "CREATE TABLE orders (id INT, customer_id INT, amount DECIMAL, order_date DATE);",
    },
    {
        "question": "List all customers who have never placed an order.",
        "schema": "CREATE TABLE customers (id INT, name TEXT); CREATE TABLE orders (id INT, customer_id INT);",
    },
    {
        "question": "What is the average salary by department?",
        "schema": "CREATE TABLE employees (id INT, name TEXT, department TEXT, salary DECIMAL);",
    },
]

for ex in examples:
    sql = generate_sql(ex["question"], ex["schema"])
    print(f"Q: {ex['question']}")
    print(f"SQL: {sql}")
    print("-" * 60)